**Importar las librerias necesarias.**

In [1]:
import requests
from bs4 import BeautifulSoup, NavigableString, Tag
import pandas as pd
import time
import re
import os
import random
import asyncio
from urllib.parse import urlparse
from playwright.async_api import async_playwright
import nest_asyncio
from requests_html import AsyncHTMLSession
import asyncio
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
from datetime import datetime

**Web scraping: El tiempo**

In [15]:
# Definir la función de scraping
def scrape_eltiempo_fentanilo_jupyter():
    base_search_url = "https://www.eltiempo.com/buscar/"
    search_params = {
        'q': 'fentanilo',
        'articleTypes': 'especial_modular,especial-tipo-d,default,gallery,video_detail,play_video_detail,premium,opinion,obituario,carta,podcast,editorial,foro,caricatura,infografia',
        'categories_ids_or': '',
        'from': '',
        'until': ''
    }

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
        'Accept-Encoding': 'gzip, deflate, br',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image:apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
        'Connection': 'keep-alive'
    }

    all_articles_data = []

    article_url_pattern = re.compile(r'eltiempo\.com/(?:[a-z0-9-]+/){1,5}[a-z0-9-]+-\d{6,}', re.IGNORECASE)
    potential_article_path_pattern = re.compile(r'/(colombia|mundo|politica|economia|justicia|cultura|deportes|bogota|archivo)/')

    # Patrón regex para la fecha visible: AAAA-MM-DD
    date_text_pattern = re.compile(r'(\d{4}-\d{2}-\d{2})') 

    # --- ¡CAMBIO CLAVE AQUÍ! Iterar hasta la página requerida ---
    for page_num in range(41, 76): 
        print(f"Scraping search results page: {page_num}/76")
        search_params['page'] = page_num
        search_url = requests.get(base_search_url, params=search_params).url

        try:
            response = requests.get(search_url, headers=headers, timeout=15) # Aumento de timeout
            response.raise_for_status() # Lanza un error si la solicitud no fue exitosa (código 4xx o 5xx)
            soup = BeautifulSoup(response.text, 'lxml')

            search_results_container = soup.find('div', class_='o-list o-list-border')

            if not search_results_container:
                print(f"No search results container found on page {page_num}. Skipping to next page.")
                continue # Continuar a la siguiente página de búsqueda

            potential_links = search_results_container.find_all('a')
            
            article_links_found_on_page = []
            for link_tag in potential_links:
                href = link_tag.get('href')
                if href:
                    full_url = href
                    if not full_url.startswith('http'):
                        full_url = f"https://www.eltiempo.com{full_url}"
                    
                    if (article_url_pattern.search(full_url) or potential_article_path_pattern.search(full_url)) and 'page=' not in full_url:
                        if link_tag.text.strip():
                            article_links_found_on_page.append(full_url)
            
            article_links_found_on_page = list(set(article_links_found_on_page)) # Eliminar duplicados

            if not article_links_found_on_page:
                print(f"No valid article URLs found on page {page_num}. Skipping to next page.")
                continue # Continuar a la siguiente página de búsqueda

            for article_url in article_links_found_on_page:
                print(f"  Processing article: {article_url}")
                # --- ¡CAMBIO CLAVE AQUÍ! Pausa aleatoria entre artículos ---
                time.sleep(random.uniform(3, 5)) # Pausa entre 3 y 5 segundos

                try:
                    article_response = requests.get(article_url, headers=headers, timeout=15) # Aumento de timeout
                    article_response.raise_for_status()
                    article_soup = BeautifulSoup(article_response.text, 'lxml')

                    # Extract Headline
                    headline_tag = article_soup.find('h1', class_='c-articulo__titulo')
                    headline = headline_tag.text.strip() if headline_tag else 'N/A'

                    # Extract Author
                    author = 'N/A'
                    author_date_container = article_soup.find('div', class_='c-articulo__autor__content')
                    if author_date_container:
                        author_tag = author_date_container.find('a', class_='c-articulo__autor__nombre')
                        if not author_tag:
                            author_tag = author_date_container.find('span', class_='c-articulo__autor__nombre')
                        if not author_tag: 
                            author_tag = author_date_container.find(['span', 'a'], text=True) 
                        author = author_tag.text.strip() if author_tag else 'N/A'

                    # --- FECHA: Lógica de extracción más robusta (basada en AAAA-MM-DD) ---
                    date = 'N/A'
                    
                    # Intenta primero extraer del atributo datetime de la etiqueta <time>
                    time_tag = article_soup.find('time')
                    if time_tag and 'datetime' in time_tag.attrs:
                        datetime_value = time_tag['datetime']
                        date_match = re.match(r'(\d{4}-\d{2}-\d{2})', datetime_value) # Coincide YYYY-MM-DD al inicio
                        if date_match:
                            date = date_match.group(1)
                    
                    # Si no se encontró en el atributo datetime, buscar por patrón de texto en las áreas relevantes
                    if date == 'N/A':
                        info_container = article_soup.find('div', class_='c-articulo__info')
                        author_content_container = article_soup.find('div', class_='c-articulo__autor__content')
                        
                        search_areas = []
                        if info_container:
                            search_areas.append(info_container)
                        if author_content_container:
                            search_areas.append(author_content_container)
                        
                        # Si no se encontró ninguno de los contenedores específicos, buscar en todo el body como último recurso
                        if not search_areas:
                            search_areas.append(article_soup.find('body')) 

                        for area in search_areas:
                            if area:
                                text_to_search = area.get_text(separator=' ', strip=True)
                                match = date_text_pattern.search(text_to_search)
                                if match:
                                    date = match.group(1) 
                                    break 
                    
                    # Limpiar cualquier texto adicional como "Actualizado" o "PUBL.:"
                    if date != 'N/A':
                        date = date.replace("Actualizado", "").strip()
                        date = date.replace("PUBL.:", "").strip()


                    # --- Extract and Clean Main Body Content from <div class="c-cuerpo"> ---
                    main_content_tag = article_soup.find('div', class_='c-cuerpo')
                    article_body_text = ''

                    if main_content_tag:
                        cloned_main_content = BeautifulSoup(str(main_content_tag), 'lxml')

                        # Remove unwanted sections:
                        for p_media_txt in cloned_main_content.find_all('p', class_='c-cuerpo__media_txt'):
                            p_media_txt.decompose()
                        
                        for lea_tambien_div in cloned_main_content.find_all('div', class_='c-leatambien'):
                            lea_tambien_div.decompose()
                        
                        for related_links_div in cloned_main_content.find_all('div', class_='related-links'):
                            related_links_div.decompose()
                        
                        smartblock_div = cloned_main_content.find('div', id='smartblock-placeholder-wrapper')
                        if smartblock_div:
                            smartblock_div.decompose()
                        
                        for twitter_embed_div in cloned_main_content.find_all('div', class_='twitter-tweet'):
                            twitter_embed_div.decompose()

                        mas_noticias_h2 = cloned_main_content.find('h2', string=re.compile(r'Más noticias', re.IGNORECASE))
                        if mas_noticias_h2:
                            current_tag = mas_noticias_h2
                            while current_tag:
                                next_tag = current_tag.next_sibling
                                current_tag.decompose()
                                current_tag = next_tag

                        paragraphs = cloned_main_content.find_all(['p', 'ul', 'ol', 'blockquote', 'div']) 
                        for p in paragraphs:
                            text_content = p.get_text(separator=' ', strip=True)
                            if text_content:
                                article_body_text += text_content + '\n'

                        article_body_text = article_body_text.strip()

                        fentanilo_count = len(re.findall(r'fentanilo', article_body_text, re.IGNORECASE))

                        if fentanilo_count >= 1:
                            all_articles_data.append({
                                'Titulo': headline,
                                'Cuerpo_Noticia': article_body_text,
                                'Autor': author,
                                'Fecha': date,
                                'URL': article_url,
                                'Conteo_Fentanilo': fentanilo_count
                            })
                        else:
                            print(f"    Skipping article (fentanilo count < 1): {headline}")
                    else:
                        print(f"    Main content (div.c-cuerpo) not found for: {article_url}. Skipping article.")
                        
                except requests.exceptions.RequestException as e:
                    print(f"  Error accessing article URL {article_url}: {e}. Skipping article.")
                    continue # Saltar al siguiente artículo si hay un error de solicitud
                except Exception as e:
                    print(f"  Error processing article {article_url}: {e}. Skipping article.")
                    continue # Saltar al siguiente artículo si hay otro tipo de error

        except requests.exceptions.RequestException as e:
            print(f"Error accessing search page {search_url}: {e}. Skipping to next search page.")
            continue # Saltar a la siguiente página de búsqueda si hay un error de solicitud
        except Exception as e:
            print(f"An unexpected error occurred on search page {search_url}: {e}. Skipping to next search page.")
            continue # Saltar a la siguiente página de búsqueda si hay otro tipo de error

        # --- ¡CAMBIO CLAVE AQUÍ! Pausa aleatoria entre páginas de búsqueda ---
        time.sleep(random.uniform(5, 10)) # Pausa entre 5 y 10 segundos

    if all_articles_data:
        df = pd.DataFrame(all_articles_data)
        output_filename = 'noticias_fentanilo_eltiempo_41_76.csv' # Nuevo nombre de archivo
        current_dir = os.getcwd()
        full_output_path = os.path.join(current_dir, output_filename)

        df.to_csv(full_output_path, index=False, encoding='utf-8-sig')
        print(f"\nScraping complete. Data saved to {full_output_path}")
        from IPython.display import display
        display(df.head())
    else:
        print("\nNo articles found matching the criteria.")


In [16]:
# Llamada a la función para ejecutar el scraping
scrape_eltiempo_fentanilo_jupyter()

Scraping search results page: 41/76
  Processing article: https://www.eltiempo.com/justicia/delitos/general-rojas-gomez-los-nexos-con-las-disidencias-por-los-que-lo-investiga-la-fiscalia-809994
  Processing article: https://www.eltiempo.com/justicia/investigacion
    Main content (div.c-cuerpo) not found for: https://www.eltiempo.com/justicia/investigacion. Skipping article.
  Processing article: https://www.eltiempo.com/justicia/cortes
    Main content (div.c-cuerpo) not found for: https://www.eltiempo.com/justicia/cortes. Skipping article.
  Processing article: https://www.eltiempo.com/justicia/conflicto-y-narcotrafico
    Main content (div.c-cuerpo) not found for: https://www.eltiempo.com/justicia/conflicto-y-narcotrafico. Skipping article.
  Processing article: https://www.eltiempo.com/mundo/eeuu-y-canada/bob-menendez-como-impacta-a-colombia-acusacion-por-corrupcion-y-salida-de-comision-809406
  Processing article: https://www.eltiempo.com/mundo/eeuu-y-canada/ee-uu-incluye-a-alias-

,Titulo,Cuerpo_Noticia,Autor,Fecha,URL,Conteo_Fentanilo
0,Los nexos con las disidencias por los que Fisc...,Hasta hace menos de un mes el general John Jai...,Sair Buitrago,2023-09-26,https://www.eltiempo.com/justicia/delitos/gene...,2
1,¿Por qué acusaciones de corrupción contra un s...,"El senador Bob Menéndez, por décadas uno de lo...",Sergio Gómez Maseri,2023-09-25,https://www.eltiempo.com/mundo/eeuu-y-canada/b...,2
2,"EE. UU. incluye a alias 'Chiquito Malo', jefe ...",Estados Unidos anunció este martes la inclusió...,SERGIO GÓMEZ MÁSERI,2023-09-26,https://www.eltiempo.com/mundo/eeuu-y-canada/e...,26
3,'Te veré mañana': madre de Angus Cloud revela ...,"""Te quiero, mamá"", fueron las últimas palabras...",Laura Natalia Bohórquez Roncancio,2023-09-27,https://www.eltiempo.com/cultura/cine-y-tv/mam...,2
4,Cómo es la cárcel-rascacielos de Chicago en la...,"Ovidio Guzmán, uno de los hijos del narcotrafi...",Redacción BBC News Mundo,2023-09-25,https://www.eltiempo.com/mundo/eeuu-y-canada/o...,2


**Web scraping: SEMANA**

In [ ]:
# --- CONFIGURACIÓN INICIAL ---
URL_BUSQUEDA = "https://www.semana.com/buscador/?query=fentanilo"
BASE_URL = "https://www.semana.com"
FECHA_INICIO = datetime(2016, 1, 1)
FECHA_FINAL = datetime(2024, 12, 31)
PALABRA_CLAVE = "fentanilo"
PAGINAS_POR_CSV = 10

def parsear_fecha(texto_fecha):
    """Convierte el texto de la fecha a un objeto datetime."""
    try:
        return datetime.strptime(texto_fecha, '%d/%m/%Y')
    except ValueError:
        return None

def obtener_cuerpo_noticia(soup):
    """
    Extrae el cuerpo de la noticia usando el selector <main>
    """
    cuerpo_articulo = soup.find('main', class_='mx-auto mb-4 max-w-[1152px] px-2.5')
    if not cuerpo_articulo:
        return ""
    
    for element in cuerpo_articulo.find_all(['figcaption', 'div'], class_=['not-prose', 'relative', 'my-2']):
        element.decompose()
    if cuerpo_articulo.find('div', id="smartblock-placeholder-image"):
        cuerpo_articulo.find('div', id="smartblock-placeholder-image").decompose()
    
    textos = []
    for p_tag in cuerpo_articulo.find_all('p'):
        texto_limpio = p_tag.text.strip()
        if texto_limpio:
            textos.append(texto_limpio)
            
    return "\n".join(textos)

def scrap_articulo(url, driver):
    """Navega a la URL de un artículo y extrae todos los detalles."""
    print(f"  > Scrapeando artículo: {url}")
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, 'article'))
        )
        soup_articulo = BeautifulSoup(driver.page_source, 'html.parser')

        # --- Extracción de datos del artículo con los nuevos selectores ---
        titulo = soup_articulo.find('h1').get_text(strip=True) if soup_articulo.find('h1') else 'No encontrado'
        
        meta_container = soup_articulo.find('div', class_='mx-auto max-w-[968px]')
        autor = 'No encontrado'
        fecha_texto = 'No encontrada'
        if meta_container:
            autor_tag = meta_container.find('div', class_='mb-5 text-smoke-500')
            autor = autor_tag.text.strip() if autor_tag else 'No encontrado'
            fecha_tag = meta_container.find('div', class_='mb-5 text-xs text-smoke-500')
            fecha_texto = fecha_tag.text.strip() if fecha_tag else 'No encontrada'
        
        cuerpo = obtener_cuerpo_noticia(soup_articulo)
        texto_completo = f"{titulo} {cuerpo}".lower()
        conteo = len(re.findall(r'\b' + PALABRA_CLAVE.lower() + r'\b', texto_completo))

        # Se eliminó la validación de fecha aquí para procesar todos los artículos.

        return {
            'título': titulo,
            'cuerpo_noticia': cuerpo,
            'autor': autor,
            'fecha': fecha_texto,
            'url': url,
            'conteo_fentanilo': conteo
        }
    except Exception as e:
        print(f"    * Error al scrapear el artículo {url}: {e}")
        return None
    finally:
        driver.back()

# --- INICIO DEL SCRIPT PRINCIPAL ---
GECKODRIVER_PATH = os.path.join(os.getcwd(), 'geckodriver.exe')
LIBREWOLF_PATH = r"C:\Program Files\LibreWolf\librewolf.exe"
print("Iniciando el navegador LibreWolf...")
options = FirefoxOptions()
options.add_argument('--headless')
options.add_argument('--log-level=3')
options.binary_location = LIBREWOLF_PATH
service = Service(GECKODRIVER_PATH)
driver = webdriver.Firefox(service=service, options=options)
articulos_recopilados = []
continuar_paginando = True
pagina_actual = 1
try:
    driver.get(URL_BUSQUEDA)
    while continuar_paginando:
        print(f"\n--- Procesando página de resultados #{pagina_actual} ---")
        WebDriverWait(driver, 20).until(
            EC.visibility_of_element_located((By.ID, 'queryly_advanced_container'))
        )
        time.sleep(3)
        soup_resultados = BeautifulSoup(driver.page_source, 'html.parser')
        resultados = soup_resultados.find_all('div', class_='queryly_item_row')
        if not resultados:
            print("No se encontraron más resultados. Finalizando.")
            break
        print(f"Se encontraron {len(resultados)} resultados en la página.")
        detener_por_fecha = False
        for item in resultados:
            meta_div = item.find('div', class_='queryly_item_meta')
            fecha_str = meta_div.get_text(strip=True) if meta_div else ''
            fecha_dt = parsear_fecha(fecha_str)
            if fecha_dt and fecha_dt < FECHA_INICIO:
                print(f"Noticia del {fecha_str} es anterior a {FECHA_INICIO.date()}. Deteniendo la recolección.")
                detener_por_fecha = True
                break
            url_articulo_tag = item.find('a', href=True)
            if url_articulo_tag:
                url_relativa = url_articulo_tag['href']
                url_articulo = f"{BASE_URL}{url_relativa}"
                datos_articulo = scrap_articulo(url_articulo, driver)
                WebDriverWait(driver, 20).until(
                    EC.visibility_of_element_located((By.ID, 'queryly_advanced_container'))
                )
                time.sleep(2)
                if datos_articulo:
                    articulos_recopilados.append(datos_articulo)
            else:
                print(f"  - No se encontró URL en el resultado. Saltando.")
        
        if pagina_actual % PAGINAS_POR_CSV == 0:
            if articulos_recopilados:
                nombre_archivo = f'noticias_fentanilo_pag_{pagina_actual - PAGINAS_POR_CSV + 1}_a_{pagina_actual}.csv'
                df = pd.DataFrame(articulos_recopilados)
                columnas_ordenadas = ['título', 'cuerpo_noticia', 'autor', 'fecha', 'url', 'conteo_fentanilo']
                df = df[columnas_ordenadas]
                df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
                print(f"\nSe guardaron {len(articulos_recopilados)} artículos en '{nombre_archivo}'.")
                articulos_recopilados = []
            else:
                print(f"\nNo se recopiló ningún artículo en las últimas {PAGINAS_POR_CSV} páginas.")
        
        if detener_por_fecha:
            continuar_paginando = False
            break
        try:
            next_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.next_btn'))
            )
            print("Haciendo clic en 'Siguiente'...")
            driver.execute_script("arguments[0].click();", next_button)
            pagina_actual += 1
        except Exception:
            print("No se encontró el botón 'Siguiente' o no es clickeable. Finalizando paginación.")
            continuar_paginando = False
finally:
    print("\nCerrando el navegador.")
    driver.quit()
if articulos_recopilados:
    print(f"\nProceso finalizado. Se recopilaron {len(articulos_recopilados)} artículos restantes.")
    df = pd.DataFrame(articulos_recopilados)
    columnas_ordenadas = ['título', 'cuerpo_noticia', 'autor', 'fecha', 'url', 'conteo_fentanilo']
    df = df[columnas_ordenadas]
    nombre_archivo = 'noticias_fentanilo_restantes.csv'
    df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
    print(f"Los datos restantes han sido guardados en el archivo '{nombre_archivo}'.")
else:
    print("\nNo se recopiló ningún artículo restante.")

Iniciando el navegador LibreWolf...

--- Procesando página de resultados #1 ---
Se encontraron 20 resultados en la página.
  > Scrapeando artículo: https://www.semana.com/mundo/noticias-estados-unidos/articulo/estados-unidos-congela-cuentas-y-veta-visas-a-aliados-del-narcotrafico-asi-arranca-la-ofensiva-total-contra-el-trafico-de-fentanilo/202503/
  > Scrapeando artículo: https://www.semana.com/mundo/noticias-estados-unidos/articulo/florida-endurece-medidas-contra-el-fentanilo-lo-que-debe-saber-sobre-la-ley-hb-259/202550/
  > Scrapeando artículo: https://www.semana.com/mundo/noticias-estados-unidos/articulo/impactantes-declaraciones-de-un-alcalde-de-california-sobre-problema-de-las-personas-sin-hogar-darles-fentanilo-gratis/202551/
  > Scrapeando artículo: https://www.semana.com/nacion/cali/articulo/desmantelan-en-cali-uno-de-los-mayores-laboratorios-de-fentanilo-y-otras-drogas-sinteticas/202512/
  > Scrapeando artículo: https://www.semana.com/nacion/regionales/articulo/policia-incauta

**Web scraping: Portafolio**

*Durante el desarrollo del script para la extracción de la información se encontraron varias barreras relacionadas con la ubicación <div> en el cuerpo del artículo, fueron pocos casos y se dió cuando se encontraba en secciones de editorial y opinión.*

In [1]:
# --- CONFIGURACIÓN INICIAL ---
URL_BASE = "https://www.portafolio.co"
URL_BUSQUEDA = "/buscar?q=fentanilo"
PALABRA_CLAVE = "fentanilo"
PAGINA_INICIO = 1
PAGINA_FINAL = 17

def limpiar_texto(soup, elemento_inicial):
    """Extrae el texto del cuerpo de la noticia y filtra contenido no deseado."""
    cuerpo = []
    for tag in elemento_inicial.children:
        # Si el elemento no es una etiqueta, lo ignoramos.
        if isinstance(tag, NavigableString):
            continue

        if isinstance(tag, Tag):
            # Solo busca las etiquetas <p> dentro del cuerpo del artículo.
            if tag.name == 'p':
                cuerpo.append(tag.get_text(strip=True))
            
            # Filtramos enlaces y secciones no deseadas.
            for a_tag in tag.find_all('a'):
                if a_tag.get_text(strip=True).lower().startswith(('vea más:', 'lea acá:', 'siga acá:', 'lea más:', 'vea también:', 'lea:')):
                    a_tag.decompose()
        
    texto_limpio = " ".join(cuerpo)
    return texto_limpio

def scrap_articulo(url, driver):
    """Navega a la URL de un artículo y extrae todos los detalles."""
    print(f"  > Scrapeando artículo: {url}")
    driver.get(url)
    try:
        try:
            close_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CLASS_NAME, 'btn-close'))
            )
            close_button.click()
            print("    > Botón 'Cerrar' detectado y clickeado.")
        except Exception:
            pass
        
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'section.main-container'))
        )
        
        soup_articulo = BeautifulSoup(driver.page_source, 'html.parser')
        
        titulo_tag = soup_articulo.find('h1', class_=re.compile(r'title tiemposBold8 titularUnderBlanco'))
        if not titulo_tag:
            print("    * Error: No se encontró la etiqueta <h1> del título.")
            return None
        titulo = titulo_tag.get_text(strip=True)
        
        autor_tag = soup_articulo.find('div', class_='signature openSansBold3')
        autor = autor_tag.get_text(strip=True) if autor_tag else 'No encontrado'

        # Nuevo selector para la fecha
        fecha_tag = soup_articulo.find('span', class_='date-time')
        fecha_texto = fecha_tag.get_text(strip=True) if fecha_tag else 'No encontrada'
        
        # Se mantiene el selector más específico para el cuerpo del artículo
        cuerpo_articulo_tag = soup_articulo.find('div', class_='article-content')
        if not cuerpo_articulo_tag:
            print("    * Error: No se encontró la etiqueta <div> del cuerpo del artículo.")
            return None
            
        cuerpo = limpiar_texto(soup_articulo, cuerpo_articulo_tag)
        
        texto_completo = f"{titulo} {cuerpo}".lower()
        conteo = len(re.findall(r'\b' + PALABRA_CLAVE.lower() + r'\b', texto_completo))
        
        return {
            'título': titulo,
            'cuerpo_noticia': cuerpo,
            'autor': autor,
            'fecha': fecha_texto,
            'url': url,
            'conteo_fentanilo': conteo
        }
    except Exception as e:
        print(f"    * Error al scrapear el artículo {url}: {e}")
        return None
    finally:
        driver.back()

# --- INICIO DEL SCRIPT PRINCIPAL ---
GECKODRIVER_PATH = os.path.join(os.getcwd(), 'geckodriver.exe')
LIBREWOLF_PATH = r"C:\Program Files\LibreWolf\librewolf.exe"
print("Iniciando el navegador LibreWolf...")
options = FirefoxOptions()
options.add_argument('--headless')
options.add_argument('--log-level=3')
options.binary_location = LIBREWOLF_PATH
service = Service(GECKODRIVER_PATH)
driver = webdriver.Firefox(service=service, options=options)

articulos_recopilados = []
continuar_paginando = True
pagina_actual = PAGINA_INICIO

try:
    driver.get(f"{URL_BASE}{URL_BUSQUEDA}&page={PAGINA_INICIO}")
    try:
        cookies_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "cookies-info__button"))
        )
        cookies_button.click()
        print("Aviso de cookies aceptado.")
    except Exception:
        print("No se encontró el aviso de cookies.")
    while continuar_paginando and pagina_actual <= PAGINA_FINAL:
        print(f"\n--- Procesando página de resultados #{pagina_actual} ---")
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CLASS_NAME, 'listing'))
        )
        time.sleep(3)
        soup_resultados = BeautifulSoup(driver.page_source, 'html.parser')
        resultados = soup_resultados.find_all('div', class_='listing')
        if not resultados:
            print("No se encontraron más resultados. Finalizando.")
            break
        print(f"Se encontraron {len(resultados)} resultados en la página.")
        for item in resultados:
            url_articulo_tag = item.find('a', href=True)
            if url_articulo_tag:
                url_articulo = url_articulo_tag['href']
                if not url_articulo.startswith('http'):
                    url_articulo = f"{URL_BASE}{url_articulo}"
                
                try:
                    datos_articulo = scrap_articulo(url_articulo, driver)
                    time.sleep(3)
                    if datos_articulo:
                        articulos_recopilados.append(datos_articulo)
                except Exception as e:
                    print(f"    * Error en el bucle principal al scrapear {url_articulo}: {e}")
                    try:
                        driver.back()
                        time.sleep(3)
                    except Exception:
                        print("No se pudo volver a la página de resultados. Se reiniciará el navegador.")
                        driver.quit()
                        driver = webdriver.Firefox(service=service, options=options)
                        driver.get(f"{URL_BASE}{URL_BUSQUEDA}&page={pagina_actual}")
                        time.sleep(5)
            else:
                print(f"  - No se encontró URL en el resultado. Saltando.")
        if pagina_actual < PAGINA_FINAL:
            try:
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.pagination ul li.next a'))
                )
                print("Haciendo clic en 'Siguiente'...")
                driver.execute_script("arguments[0].click();", next_button)
                pagina_actual += 1
            except Exception:
                print("No se encontró el botón 'Siguiente' o no es clickeable. Finalizando paginación.")
                continuar_paginando = False
        else:
            print("Se ha alcanzado la última página. Finalizando.")
            continuar_paginando = False
finally:
    print("\nCerrando el navegador.")
    driver.quit()
if articulos_recopilados:
    print(f"\nProceso finalizado. Se recopilaron {len(articulos_recopilados)} artículos.")
    df = pd.DataFrame(articulos_recopilados)
    df.to_csv('noticias_fentanilo.csv', index=False, encoding='utf-8-sig')
    print("Los datos han sido guardados en el archivo 'noticias_fentanilo.csv'.")
else:
    print("\nNo se recopiló ningún artículo.")

NameError: name 'os' is not defined

**Web scraping: El Universal**

In [8]:
# --- CONFIGURACIÓN INICIAL ---
URL_BASE = "https://www.eluniversal.com.co"
URL_BUSQUEDA = "/buscador/?query=fentanilo"
PALABRA_CLAVE = "fentanilo"
PAGINA_INICIO = 1
PAGINA_FINAL = 10  

def limpiar_texto(soup_articulo):
    """Extrae el texto del cuerpo de la noticia y filtra contenido no deseado."""
    
    # Encontrar el div principal del cuerpo
    cuerpo_articulo_tag = soup_articulo.find('div', class_=re.compile(r'eu-layoutArticle__LayoutArticleBody-sc-j27yzk-3'))
    
    if not cuerpo_articulo_tag:
        return ""

    # Eliminar el contenido de la etiqueta <aside>
    aside_tag = cuerpo_articulo_tag.find('aside', class_=re.compile(r'eu-newsRanking__StyledNewsRankig-sc-1l2rlkc-0'))
    if aside_tag:
        aside_tag.extract()

    # Extraer todo el texto de los párrafos dentro del cuerpo
    cuerpo = [p.get_text(strip=True) for p in cuerpo_articulo_tag.find_all('p')]
    texto_limpio = " ".join(cuerpo)

    # Eliminar el texto del subtítulo de la imagen si se encuentra
    figcaption = soup_articulo.find('p', class_=re.compile(r'eu-articleHeadline__StyledArticleFigcaption-sc'))
    if figcaption and texto_limpio.startswith(figcaption.get_text(strip=True)):
        texto_limpio = texto_limpio[len(figcaption.get_text(strip=True)):].strip()
        
    return texto_limpio

def scrap_articulo(url, driver):
    """Navega a la URL de un artículo y extrae todos los detalles."""
    print(f"  > Scrapeando artículo: {url}")
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'h1.eu-article-headline'))
        )
        
        # Obtener el HTML completo una vez que la página está cargada
        soup_articulo = BeautifulSoup(driver.page_source, 'html.parser')

        titulo_tag = soup_articulo.find('h1', class_=re.compile(r'eu-articleHeadline__StyledArticleHeadlineTitle-sc'))
        titulo = titulo_tag.get_text(strip=True) if titulo_tag else 'No encontrado'
        
        autor_tag = soup_articulo.find('a', class_=re.compile(r'eu-articleHeadline__StyledArticleAuthors'))
        autor = autor_tag.get_text(strip=True) if autor_tag else 'No encontrado'
        
        fecha_tag = soup_articulo.find('div', class_=re.compile(r'eu-date-time-block'))
        fecha_texto = fecha_tag.get_text(strip=True) if fecha_tag else 'No encontrada'
        
        cuerpo = limpiar_texto(soup_articulo)
        
        texto_completo = f"{titulo} {cuerpo}".lower()
        conteo = len(re.findall(r'\b' + PALABRA_CLAVE.lower() + r'\b', texto_completo))
        
        return {
            'título': titulo,
            'cuerpo_noticia': cuerpo,
            'autor': autor,
            'fecha': fecha_texto,
            'url': url,
            'conteo_fentanilo': conteo
        }
    except Exception as e:
        print(f"    * Error al scrapear el artículo {url}: {e}")
        return None
    finally:
        driver.back()

# --- INICIO DEL SCRIPT PRINCIPAL ---
GECKODRIVER_PATH = os.path.join(os.getcwd(), 'geckodriver.exe')
LIBREWOLF_PATH = r"C:\Program Files\LibreWolf\librewolf.exe"
print("Iniciando el navegador LibreWolf...")
options = FirefoxOptions()
options.add_argument('--headless')
options.add_argument('--log-level=3')
options.binary_location = LIBREWOLF_PATH
service = Service(GECKODRIVER_PATH)
driver = webdriver.Firefox(service=service, options=options)

articulos_recopilados = []
continuar_paginando = True
pagina_actual = PAGINA_INICIO

try:
    driver.get(f"{URL_BASE}{URL_BUSQUEDA}")
    
    while continuar_paginando and pagina_actual <= PAGINA_FINAL:
        print(f"\n--- Procesando página de resultados #{pagina_actual} ---")
        
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.ID, 'resultdata'))
        )
        time.sleep(3) 
        
        soup_resultados = BeautifulSoup(driver.page_source, 'html.parser')
        resultados_container = soup_resultados.find('div', id='resultdata')
        resultados = resultados_container.find_all('div', class_='queryly_item_row')
        
        if not resultados:
            print("No se encontraron más resultados. Finalizando.")
            break
        
        print(f"Se encontraron {len(resultados)} resultados en la página.")
        for item in resultados:
            url_articulo_tag = item.find('a', href=True)
            if url_articulo_tag:
                url_articulo = url_articulo_tag['href']
                if not url_articulo.startswith('http'):
                    url_articulo = f"{URL_BASE}{url_articulo}"
                
                try:
                    datos_articulo = scrap_articulo(url_articulo, driver)
                    time.sleep(3)
                    if datos_articulo:
                        articulos_recopilados.append(datos_articulo)
                except Exception as e:
                    print(f"    * Error en el bucle principal al scrapear {url_articulo}: {e}")
                    try:
                        driver.back()
                        time.sleep(3)
                    except Exception:
                        print("No se pudo volver a la página de resultados. Se reiniciará el navegador.")
                        driver.quit()
                        driver = webdriver.Firefox(service=service, options=options)
                        driver.get(f"{URL_BASE}{URL_BUSQUEDA}")
                        time.sleep(5)
            else:
                print(f"  - No se encontró URL en el resultado. Saltando.")
        
        if pagina_actual < PAGINA_FINAL:
            try:
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.next_btn'))
                )
                print("Haciendo clic en 'Siguiente'...")
                driver.execute_script("arguments[0].click();", next_button)
                pagina_actual += 1
            except Exception:
                print("No se encontró el botón 'Siguiente' o no es clickeable. Finalizando paginación.")
                continuar_paginando = False
        else:
            print("Se ha alcanzado la última página. Finalizando.")
            continuar_paginando = False
finally:
    print("\nCerrando el navegador.")
    driver.quit()

if articulos_recopilados:
    print(f"\nProceso finalizado. Se recopilaron {len(articulos_recopilados)} artículos.")
    df = pd.DataFrame(articulos_recopilados)
    df.to_csv('noticias_fentanilo_el_universal.csv', index=False, encoding='utf-8-sig')
    print("Los datos han sido guardados en el archivo 'noticias_fentanilo_el_universal.csv'.")
else:
    print("\nNo se recopiló ningún artículo.")

Iniciando el navegador LibreWolf...

--- Procesando página de resultados #1 ---
Se encontraron 20 resultados en la página.
  > Scrapeando artículo: https://www.eluniversal.com.co/mundo/2025/04/23/darles-fentanilo-gratis-la-propuesta-de-un-alcalde-para-los-indigentes/
  > Scrapeando artículo: https://www.eluniversal.com.co/colombia/2025/04/03/estudiantes-del-sena-intoxicados-por-presunto-consumo-de-fentanilo-en-gomitas/
  > Scrapeando artículo: https://www.eluniversal.com.co/palante-chamos-y-chamas/2024/05/27/alertan-sobre-adiccion-al-fentanilo-entre-migrantes-de-la-frontera-de-mexico-con-eeuu/
  > Scrapeando artículo: https://www.eluniversal.com.co/cartagena/2023/10/24/que-es-la-naloxona-y-como-funciona-contra-el-fentanilo/
  > Scrapeando artículo: https://www.eluniversal.com.co/mundo/2023/12/04/eeuu-lanza-una-fuerza-de-ataque-para-luchar-contra-el-fentanilo/
  > Scrapeando artículo: https://www.eluniversal.com.co/colombia/2023/08/15/el-fentanilo-no-esta-penalizado-director-antinarcoti

**Web scraping: Vanguardia**

*La estructura de su página web no permitió que se extrajeran la mayoría de los nombres de los autores de las notas.*

In [ ]:
# --- CONFIGURACIÓN INICIAL ---
URL_BASE = "https://www.vanguardia.com"
URL_BUSQUEDA = "/buscador/?query=fentanilo"
PALABRA_CLAVE = "fentanilo"
PAGINA_INICIO = 1
PAGINA_FINAL = 15 # Puedes ajustar este valor si quieres un límite
TIMEOUT = 30 # Definición de la variable TIMEOUT al inicio para que sea accesible globalmente

def limpiar_texto(soup_articulo):
    """Extrae el texto del cuerpo de la noticia y filtra contenido no deseado."""
    cuerpo_principal_tag = soup_articulo.find('article', class_=re.compile(r'vg-articleBody__Article-sc'))
    if not cuerpo_principal_tag:
        return ""
    elementos_a_omitir = [
        'section.vg-articleBody__ImageCtn-sc',
        'div.vg-storyCardBase__StoryCardBase-sc',
        'aside.vg-newsRanking__StyledNewsRanking-sc',
        'section.vg-slider-rightrail__StyledSliderRightRail-sc',
        'div.vg-multimediaChain__StyledMainContainer-sc',
        'div.vg-recomendedCard__StyledRecomendedCardContainer-sc',
        'p.vg-articleBody__Paragraph-sc-1payatl-2'
    ]
    for selector in elementos_a_omitir:
        tag_match = cuerpo_principal_tag.find(re.compile(selector.split('.')[0]), class_=re.compile(selector.split('.')[1]))
        if tag_match:
            tag_match.extract()
    final_tag = cuerpo_principal_tag.find('div', id='imgsuscribetepie')
    if final_tag:
        for tag in final_tag.find_all_previous():
            tag.extract()
    cuerpo = []
    for p_tag in cuerpo_principal_tag.find_all('p'):
        p_text = p_tag.get_text(strip=True)
        if p_text.startswith(('Se recomienda:', 'Le puede interesar:', 'También:')):
            continue
        cuerpo.append(p_text)
    texto_limpio = " ".join(cuerpo)
    return texto_limpio

def scrap_articulo(url, driver):
    """Navega a la URL de un artículo y extrae todos los detalles."""
    print(f"  > Scrapeando artículo: {url}")
    try:
        driver.get(url)
        WebDriverWait(driver, TIMEOUT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'h1.vg-article-headline'))
        )
        
        try:
            titulo_element = driver.find_element(By.CSS_SELECTOR, 'h1.vg-article-headline')
            titulo = titulo_element.text if titulo_element else 'No encontrado'
        except NoSuchElementException:
            titulo = 'No encontrado'
        
        autor = 'No encontrado'
        selectores_autor = [
            'div.vg-articleBody__StyledArticleBylineFP-sc-1payatl-11',
            'div.vg-author-byline',
            'div.byline__author',
            'a.vg-byline',
            'span.vg-byline'
        ]
        for selector in selectores_autor:
            try:
                autor_element = driver.find_element(By.CSS_SELECTOR, selector)
                autor = autor_element.text
                break
            except NoSuchElementException:
                continue
        
        if autor == 'No encontrado':
            soup_respaldo = BeautifulSoup(driver.page_source, 'html.parser')
            autor_tag = soup_respaldo.find('div', class_=re.compile(r'vg-articleBody__StyledArticleBylineFP-sc-'))
            if autor_tag and 'Por:' in autor_tag.get_text():
                texto_autor = autor_tag.get_text(strip=True)
                autor = texto_autor.split('Por:')[1].strip()
        
        try:
            fecha_element = driver.find_element(By.CSS_SELECTOR, 'div.vg-date-time-block span')
            fecha_texto = fecha_element.text if fecha_element else 'No encontrada'
        except NoSuchElementException:
            fecha_texto = 'No encontrada'

        soup_articulo = BeautifulSoup(driver.page_source, 'html.parser')
        cuerpo = limpiar_texto(soup_articulo)
        
        texto_completo = f"{titulo} {cuerpo}".lower()
        conteo = len(re.findall(r'\b' + PALABRA_CLAVE.lower() + r'\b', texto_completo))
        
        return {
            'título': titulo,
            'cuerpo_noticia': cuerpo,
            'autor': autor,
            'fecha': fecha_texto,
            'url': url,
            'conteo_fentanilo': conteo
        }
    except TimeoutException:
        print(f"    * Error de tiempo de espera al cargar el artículo: {url}. Omitiendo.")
        return None
    except Exception as e:
        print(f"    * Error al scrapear el artículo {url}: {e}")
        return None
    finally:
        try:
            driver.back()
        except:
            pass

# --- INICIO DEL SCRIPT PRINCIPAL ---
GECKODRIVER_PATH = os.path.join(os.getcwd(), 'geckodriver.exe')
LIBREWOLF_PATH = r"C:\Program Files\LibreWolf\librewolf.exe"
print("Iniciando el navegador LibreWolf...")
options = FirefoxOptions()
options.add_argument('--headless')
options.add_argument('--log-level=3')
options.binary_location = LIBREWOLF_PATH
service = Service(GECKODRIVER_PATH)
driver = webdriver.Firefox(service=service, options=options)

articulos_recopilados = []
urls_vistas = set()
continuar_paginando = True
pagina_actual = PAGINA_INICIO

try:
    driver.get(f"{URL_BASE}{URL_BUSQUEDA}")
    
    while continuar_paginando and pagina_actual <= PAGINA_FINAL:
        print(f"\n--- Procesando página de resultados #{pagina_actual} ---")
        
        try:
            WebDriverWait(driver, TIMEOUT).until(
                EC.presence_of_element_located((By.ID, 'resultdata'))
            )
            time.sleep(3) 
            
            soup_resultados = BeautifulSoup(driver.page_source, 'html.parser')
            resultados_container = soup_resultados.find('div', id='resultdata')
            resultados = resultados_container.find_all('div', class_='queryly_item_row')
            
            if not resultados:
                print("No se encontraron más resultados. Finalizando.")
                break
            
            print(f"Se encontraron {len(resultados)} resultados en la página.")
            for item in resultados:
                url_articulo_tag = item.find('a', href=True)
                if url_articulo_tag:
                    url_articulo = url_articulo_tag['href']
                    if not url_articulo.startswith('http'):
                        url_articulo = f"{URL_BASE}{url_articulo}"

                    if url_articulo not in urls_vistas:
                        urls_vistas.add(url_articulo)
                        datos_articulo = scrap_articulo(url_articulo, driver)
                        time.sleep(3)
                        if datos_articulo:
                            articulos_recopilados.append(datos_articulo)
                    else:
                        print(f"  - URL duplicada detectada: {url_articulo}. Omitiendo.")
                else:
                    print(f"  - No se encontró URL en el resultado. Saltando.")
        
        except TimeoutException:
            print("Error de tiempo de espera al cargar la página de resultados. Intentando continuar.")
        
        # Guardar en CSV cada 5 páginas
        if pagina_actual % 5 == 0 and articulos_recopilados:
            df = pd.DataFrame(articulos_recopilados)
            df.to_csv(f'noticias_fentanilo_vanguardia_pag_{pagina_actual}.csv', index=False, encoding='utf-8-sig')
            print(f"\nSe han guardado los datos de las últimas 5 páginas en el archivo 'noticias_fentanilo_vanguardia_pag_{pagina_actual}.csv'.")
            articulos_recopilados = [] # Reiniciar la lista para las siguientes 5 páginas

        if pagina_actual < PAGINA_FINAL:
            try:
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.next_btn'))
                )
                print("Haciendo clic en 'Siguiente'...")
                driver.execute_script("arguments[0].click();", next_button)
                pagina_actual += 1
            except Exception:
                print("No se encontró el botón 'Siguiente' o no es clickeable. Finalizando paginación.")
                continuar_paginando = False
        else:
            print("Se ha alcanzado la última página. Finalizando.")
            continuar_paginando = False
finally:
    print("\nCerrando el navegador.")
    driver.quit()

if articulos_recopilados:
    df = pd.DataFrame(articulos_recopilados)
    df.to_csv('noticias_fentanilo_vanguardia_final.csv', index=False, encoding='utf-8-sig')
    print("Los datos restantes han sido guardados en el archivo 'noticias_fentanilo_vanguardia_final.csv'.")
elif not articulos_recopilados:
    print("\nNo se recopiló ningún artículo en el último ciclo.")

print("\nProceso finalizado.")

Iniciando el navegador LibreWolf...

--- Procesando página de resultados #1 ---
Se encontraron 20 resultados en la página.
  > Scrapeando artículo: https://www.vanguardia.com/mundo/2025/05/27/alarma-en-argentina-33-muertes-por-fentanilo-contaminado-desatan-crisis-sanitaria/
  > Scrapeando artículo: https://www.vanguardia.com/mundo/2025/05/15/peru-refuerza-medidas-contra-el-de-fentanilo-penas-de-hasta-25-anos-de-carcel/
  > Scrapeando artículo: https://www.vanguardia.com/mundo/2025/05/15/nina-de-tres-anos-casi-muere-por-fentanilo-su-madre-lo-compro-ilegalmente/
  > Scrapeando artículo: https://www.vanguardia.com/opinion/columnistas/2025/08/27/trump-esta-desorientado-en-el-control-del-narcotrafico/
  > Scrapeando artículo: https://www.vanguardia.com/mundo/2024/12/30/sheinbaum-tacha-de-poco-creible-el-reportaje-del-the-new-york-times-sobre-el-fentanilo/
  > Scrapeando artículo: https://www.vanguardia.com/mundo/2025/08/27/eeuu-incauta-un-record-de-drogas-capaz-de-causar-sobredosis-a-toda-f

**Web scraping: El país**

*A diferencia de SEMANA en este medio no se pudo acceder a cierto contenido premium o pago.*

In [ ]:
# --- CONFIGURACIÓN INICIAL ---
URL_BASE = "https://elpais.com"
URL_BUSQUEDA = "/buscador/fentanilo/"
PALABRA_CLAVE = "fentanilo"
PAGINA_INICIO = 1
PAGINA_FINAL = 35  # Límite de páginas
TIMEOUT = 30 

def limpiar_texto(soup_articulo):
    """Extrae el texto del cuerpo de la noticia y filtra contenido no deseado."""
    cuerpo_principal_tag = soup_articulo.find('article', id='main-content')
    if not cuerpo_principal_tag:
        return ""

    cuerpo_seccion = cuerpo_principal_tag.find('section', class_=re.compile(r'a_c'))
    if not cuerpo_seccion:
        cuerpo_seccion = cuerpo_principal_tag
    
    stop_tags = cuerpo_seccion.find_all(['section', 'aside'], class_=['w w-sea', 'w w-fid'])
    if stop_tags:
        for stop_tag in stop_tags:
            for sibling in list(stop_tag.find_next_siblings()) + [stop_tag]:
                sibling.extract()
    
    # Nuevo filtro: Eliminar el contenedor completo del paywall de suscripción
    paywall_containers = cuerpo_seccion.find_all('div', class_=re.compile(r'a_s _cf|a_r _cf'))
    for container in paywall_containers:
        container.extract()
    
    cuerpo = []
    for elem in cuerpo_seccion.find_all(['p', 'h2', 'h3', 'blockquote']):
        text = elem.get_text(strip=True)
        if text:
            cuerpo.append(text)
    
    texto_limpio = " ".join(cuerpo)
    return texto_limpio

def scrap_articulo(url, driver):
    """Navega a la URL de un artículo y extrae todos los detalles."""
    print(f"  > Scrapeando artículo: {url}")
    try:
        driver.get(url)
        WebDriverWait(driver, TIMEOUT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'article#main-content'))
        )

        soup_articulo = BeautifulSoup(driver.page_source, 'html.parser')

        if soup_articulo.find('div', id='ctn_freemium_article') or soup_articulo.find('div', id='ctn_premium_article'):
            print(f"    - Artículo omitido (freemium/premium): {url}")
            return None

        titulo_tag = soup_articulo.find('h1', class_='a_t')
        titulo = titulo_tag.get_text(strip=True) if titulo_tag else 'No encontrado'
        
        autor_tag = soup_articulo.find('div', class_='a_md_a')
        autor = autor_tag.get_text(strip=True) if autor_tag else 'No encontrado'
        
        fecha_tag = soup_articulo.find('div', class_='a_md_f')
        fecha = fecha_tag.get_text(strip=True) if fecha_tag else 'No encontrada'
        
        cuerpo = limpiar_texto(soup_articulo)
        
        texto_completo = f"{titulo} {cuerpo}".lower()
        conteo = len(re.findall(r'\b' + PALABRA_CLAVE.lower() + r'\b', texto_completo))
        
        return {
            'título': titulo,
            'cuerpo_noticia': cuerpo,
            'autor': autor,
            'fecha': fecha,
            'url': url,
            'conteo_fentanilo': conteo
        }
    except TimeoutException:
        print(f"    * Error de tiempo de espera al cargar el artículo: {url}. Omitiendo.")
        return None
    except Exception as e:
        print(f"    * Error al scrapear el artículo {url}: {e}")
        return None
    finally:
        try:
            driver.back()
        except:
            pass

# --- INICIO DEL SCRIPT PRINCIPAL ---
GECKODRIVER_PATH = os.path.join(os.getcwd(), 'geckodriver.exe')
LIBREWOLF_PATH = r"C:\Program Files\LibreWolf\librewolf.exe"
print("Iniciando el navegador LibreWolf...")
options = FirefoxOptions()
options.add_argument('--headless')
options.add_argument('--log-level=3')
options.binary_location = LIBREWOLF_PATH
service = Service(GECKODRIVER_PATH)
driver = webdriver.Firefox(service=service, options=options)

articulos_recopilados = []
urls_vistas = set()
pagina_actual = PAGINA_INICIO

try:
    while pagina_actual <= PAGINA_FINAL:
        url_paginada = f"{URL_BASE}{URL_BUSQUEDA}{pagina_actual}/" if pagina_actual > 1 else f"{URL_BASE}{URL_BUSQUEDA}"
        print(f"\n--- Procesando página de resultados #{pagina_actual} ---")
        
        driver.get(url_paginada)
        try:
            WebDriverWait(driver, TIMEOUT).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div.b-au_f._df'))
            )
            time.sleep(3)
            
            soup_resultados = BeautifulSoup(driver.page_source, 'html.parser')
            resultados = soup_resultados.find_all('article', class_=re.compile(r'c c-d _g _g-md c-m-l'))
            
            if not resultados:
                print("No se encontraron más resultados. Finalizando.")
                break
            
            print(f"Se encontraron {len(resultados)} resultados en la página.")
            for item in resultados:
                url_articulo_tag = item.find('a', href=True)
                if url_articulo_tag:
                    url_articulo = url_articulo_tag['href']
                    if not url_articulo.startswith('http'):
                        url_articulo = f"{URL_BASE}{url_articulo}"

                    if url_articulo not in urls_vistas:
                        urls_vistas.add(url_articulo)
                        datos_articulo = scrap_articulo(url_articulo, driver)
                        time.sleep(3)
                        if datos_articulo:
                            articulos_recopilados.append(datos_articulo)
                    else:
                        print(f"  - URL duplicada detectada: {url_articulo}. Omitiendo.")
                else:
                    print(f"  - No se encontró URL en el resultado. Saltando.")
        
        except TimeoutException:
            print("Error de tiempo de espera al cargar la página de resultados. Intentando continuar.")
        
        if pagina_actual % 10 == 0 and articulos_recopilados:
            df = pd.DataFrame(articulos_recopilados)
            df.to_csv(f'noticias_fentanilo_elpais_pag_{pagina_actual}.csv', index=False, encoding='utf-8-sig')
            print(f"\nSe han guardado los datos de las últimas 10 páginas en el archivo 'noticias_fentanilo_elpais_pag_{pagina_actual}.csv'.")
            articulos_recopilados = []

        pagina_actual += 1
finally:
    print("\nCerrando el navegador.")
    driver.quit()

if articulos_recopilados:
    df = pd.DataFrame(articulos_recopilados)
    df.to_csv('noticias_fentanilo_elpais_final.csv', index=False, encoding='utf-8-sig')
    print("Los datos restantes han sido guardados en el archivo 'noticias_fentanilo_elpais_final.csv'.")
elif not os.path.exists('noticias_fentanilo_elpais_final.csv') and not [f for f in os.listdir() if f.startswith('noticias_fentanilo_elpais_pag_')]:
    print("\nNo se recopiló ningún artículo.")

print("\nProceso finalizado.")

Iniciando el navegador LibreWolf...

--- Procesando página de resultados #1 ---
Se encontraron 20 resultados en la página.
  > Scrapeando artículo: https://elpais.com/internacional/2025-08-08/trump-ordena-al-ejercito-de-estados-unidos-combatir-a-los-carteles-de-la-droga-extranjeros.html
  > Scrapeando artículo: https://elpais.com/america/2025-08-09/la-orden-de-trump-de-usar-la-fuerza-contra-los-carteles-extranjeros-pone-en-alerta-a-maduro-y-a-mexico.html
  > Scrapeando artículo: https://elpais.com/mexico/2025-07-31/trump-amplia-durante-90-dias-mas-los-aranceles-actuales-a-mexico.html?autoplay=1
  > Scrapeando artículo: https://elpais.com/america/2025-08-28/brasil-golpea-al-crimen-organizado-con-una-megaoperacion-contra-su-negocio-energetico.html
  > Scrapeando artículo: https://elpais.com/espana/2025-03-04/tres-muertes-por-sobredosis-destapan-el-trafico-ilegal-de-medicamentos-con-fentanilo-en-menorca.html
  > Scrapeando artículo: https://elpais.com/sociedad/2025-05-14/las-muertes-por-s

**Web scraping: La patria**

In [9]:
# --- CONFIGURACIÓN INICIAL ---
URL_BASE = "https://www.lapatria.com/"
URL_BUSQUEDA = "https://www.lapatria.com/buscar#gsc.tab=0&gsc.q=fentanilo&gsc.sort=date&gsc.page="
PALABRA_CLAVE = "fentanilo"
PAGINA_INICIO = 1
PAGINA_FINAL = 10
TIMEOUT = 30

def limpiar_texto(soup_articulo, es_archivo=False):
    """Extrae y limpia el texto del cuerpo de la noticia."""
    
    # Extraer todo el texto del cuerpo de la página para un enfoque más robusto
    full_text = soup_articulo.body.get_text(separator=' ', strip=True)

    # Definir patrones para delimitar el inicio y el fin del artículo
    start_pattern = None
    end_pattern = None

    if not es_archivo:
        titulo_tag = soup_articulo.find('h1', class_='title')
        autor_tag = soup_articulo.find('div', class_='field--name-field-autor')
        fecha_tag = soup_articulo.find('div', class_='field--name-field-fecha-hora')
        
        if titulo_tag:
            start_pattern = titulo_tag.get_text(strip=True)
        if autor_tag:
            end_pattern = autor_tag.get_text(strip=True)
        if fecha_tag:
            start_pattern = fecha_tag.get_text(strip=True)
        
    else: # Lógica para URLs de archivo
        titulo_tag = soup_articulo.find('h1', id='page-title')
        autor_tag = soup_articulo.find('div', class_='field-name-body')
        fecha_tag = soup_articulo.find('span', class_='date-display-single')

        if titulo_tag:
            start_pattern = titulo_tag.get_text(strip=True)
        if autor_tag:
            autor_p = autor_tag.find('p')
            if autor_p:
                end_pattern = autor_p.get_text(strip=True)
        if fecha_tag:
            start_pattern = fecha_tag.get_text(strip=True)

    cuerpo_texto = full_text

    # Si se encuentran los patrones, se usa la extracción delimitada
    if start_pattern and end_pattern and start_pattern in full_text and end_pattern in full_text:
        start_index = full_text.find(start_pattern) + len(start_pattern)
        end_index = full_text.find(end_pattern)
        if end_index > start_index:
            cuerpo_texto = full_text[start_index:end_index]
        else: # Si el orden de patrones es incorrecto, intentar invertirlo
            end_index = full_text.find(start_pattern)
            start_index = full_text.find(end_pattern) + len(end_pattern)
            if end_index > start_index:
                cuerpo_texto = full_text[start_index:end_index]

    # Limpieza final de patrones comunes y texto no deseado
    patterns_to_remove = [
        r'Le puede interesar:.*?(\.|$)',
        r'Además:.*?(\.|$)',
        r'Vea también|Lea aquí|Siga leyendo|Entérese',
        r'Haga.*',
        r'<u>.*?</u>',
    ]

    for pattern in patterns_to_remove:
        cuerpo_texto = re.sub(pattern, '', cuerpo_texto, flags=re.DOTALL | re.IGNORECASE)

    # Eliminar texto de navegación y cabeceras
    cuerpo_texto = re.sub(r'Pasar al contenido principal.*Lunes.*Pagos en línea', '', cuerpo_texto, flags=re.DOTALL | re.IGNORECASE)
    cuerpo_texto = re.sub(r'MÁS DE CIEN AÑOS DE VERDAD.*?Navegación principal', '', cuerpo_texto, flags=re.DOTALL | re.IGNORECASE)
    cuerpo_texto = re.sub(r'Creer en Caldas.*?Internacional', '', cuerpo_texto, flags=re.DOTALL | re.IGNORECASE)
    cuerpo_texto = re.sub(r'Foto \| EFE \| LA PATRIA.*?Facebook Twitter WhatsApp', '', cuerpo_texto, flags=re.DOTALL | re.IGNORECASE)
    
    return re.sub(r'\s+', ' ', cuerpo_texto).strip()

def scrap_articulo(url, driver):
    """Navega a la URL de un artículo y extrae todos los detalles."""
    print(f"  > Scrapeando artículo: {url}")
    try:
        driver.get(url)
        
        es_archivo = 'archivo.lapatria.com' in url
        
        # Esperar que el contenedor principal cargue
        if not es_archivo:
            WebDriverWait(driver, TIMEOUT).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div.content h1.title'))
            )
            time.sleep(2)
            soup_articulo = BeautifulSoup(driver.page_source, 'html.parser')
            
            # Extracción de datos para páginas normales
            titulo_tag = soup_articulo.find('h1', class_='title')
            titulo = titulo_tag.get_text(strip=True) if titulo_tag else 'No encontrado'
            
            autor_tag = soup_articulo.find('div', class_='field--name-field-autor')
            autor = autor_tag.get_text(strip=True) if autor_tag else 'No encontrado'
            
            fecha_tag = soup_articulo.find('div', class_='field--name-field-fecha-hora')
            fecha = fecha_tag.get_text(strip=True) if fecha_tag else 'No encontrada'
            
        else: # Lógica para páginas de archivo
            WebDriverWait(driver, TIMEOUT).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'h1#page-title'))
            )
            time.sleep(2)
            soup_articulo = BeautifulSoup(driver.page_source, 'html.parser')
            
            # Extracción de datos para páginas de archivo
            titulo_tag = soup_articulo.find('h1', id='page-title')
            titulo = titulo_tag.get_text(strip=True) if titulo_tag else 'No encontrado'
            
            autor_tag = soup_articulo.find('div', class_='field-name-body')
            autor = autor_tag.find('p').get_text(strip=True) if autor_tag and autor_tag.find('p') else 'No encontrado'
            
            fecha_tag = soup_articulo.find('span', class_='date-display-single')
            fecha = fecha_tag.get_text(strip=True) if fecha_tag else 'No encontrada'

        cuerpo = limpiar_texto(soup_articulo, es_archivo)
        
        texto_completo = f"{titulo} {cuerpo}".lower()
        conteo = len(re.findall(r'\b' + PALABRA_CLAVE.lower() + r'\b', texto_completo))

        return {
            'título': titulo,
            'cuerpo_noticia': cuerpo,
            'autor': autor,
            'fecha': fecha,
            'url': url,
            'conteo_fentanilo': conteo
        }
    except TimeoutException:
        print(f"    * Error de tiempo de espera al cargar el artículo: {url}. Omitiendo.")
        return None
    except Exception as e:
        print(f"    * Error al scrapear el artículo {url}: {e}")
        return None

# --- INICIO DEL SCRIPT PRINCIPAL ---
GECKODRIVER_PATH = os.path.join(os.getcwd(), 'geckodriver.exe')
LIBREWOLF_PATH = r"C:\Program Files\LibreWolf\librewolf.exe"
print("Iniciando el navegador LibreWolf...")
options = FirefoxOptions()
options.add_argument('--headless')
options.add_argument('--log-level=3')
options.binary_location = LIBREWOLF_PATH
service = Service(GECKODRIVER_PATH)
driver = webdriver.Firefox(service=service, options=options)

articulos_recopilados = []
urls_vistas = set()

try:
    for pagina_actual in range(PAGINA_INICIO, PAGINA_FINAL + 1):
        url_paginada = f"{URL_BUSQUEDA}{pagina_actual}"
        print(f"\n--- Procesando página de resultados #{pagina_actual} ---")
        
        driver.get(url_paginada)
        try:
            WebDriverWait(driver, TIMEOUT).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div.gsc-results-wrapper-visible'))
            )
            time.sleep(3)
            
            soup_resultados = BeautifulSoup(driver.page_source, 'html.parser')
            resultados = soup_resultados.find_all('div', class_='gsc-webResult')
            
            if not resultados:
                print("No se encontraron más resultados. Finalizando.")
                break
            
            print(f"Se encontraron {len(resultados)} resultados en la página.")
            for item in resultados:
                url_articulo_tag = item.find('a', class_='gs-title')
                if url_articulo_tag and 'href' in url_articulo_tag.attrs:
                    url_articulo = url_articulo_tag['href']

                    # Filtrar URLs de sección y limpiar parámetros
                    if '?' in url_articulo:
                        url_articulo = url_articulo.split('?')[0]
                    
                    if any(s in url_articulo for s in ['/nacional', '/noticias-de-hoy', '/tags']):
                        print(f"  - URL de sección detectada: {url_articulo}. Omitiendo.")
                        continue
                    
                    if url_articulo not in urls_vistas:
                        urls_vistas.add(url_articulo)
                        datos_articulo = scrap_articulo(url_articulo, driver)
                        time.sleep(3)
                        if datos_articulo:
                            articulos_recopilados.append(datos_articulo)
                    else:
                        print(f"  - URL duplicada detectada: {url_articulo}. Omitiendo.")
                else:
                    print("  - No se encontró URL en el resultado. Saltando.")
        
        except TimeoutException:
            print("Error de tiempo de espera al cargar la página de resultados. Intentando continuar.")
        
        if pagina_actual % 5 == 0 and articulos_recopilados:
            df = pd.DataFrame(articulos_recopilados)
            df.to_csv(f'noticias_fentanilo_lapatria_pag_{pagina_actual}.csv', index=False, encoding='utf-8-sig')
            print(f"\nSe han guardado los datos de las últimas 5 páginas en el archivo 'noticias_fentanilo_lapatria_pag_{pagina_actual}.csv'.")
            articulos_recopilados = []

finally:
    print("\nCerrando el navegador.")
    driver.quit()

if articulos_recopilados:
    df = pd.DataFrame(articulos_recopilados)
    df.to_csv('noticias_fentanilo_lapatria_final.csv', index=False, encoding='utf-8-sig')
    print("Los datos restantes han sido guardados en el archivo 'noticias_fentanilo_lapatria_final.csv'.")
elif not os.path.exists('noticias_fentanilo_lapatria_final.csv') and not [f for f in os.listdir() if f.startswith('noticias_fentanilo_lapatria_pag_')]:
    print("\nNo se recopiló ningún artículo.")

print("\nProceso finalizado.")

Iniciando el navegador LibreWolf...

--- Procesando página de resultados #1 ---
Se encontraron 11 resultados en la página.
  > Scrapeando artículo: https://www.lapatria.com/internacional/8-anos-despues-condenan-responsables-de-la-muerte-de-41-ninas-en-un-incendio-aun-hay
  - URL duplicada detectada: https://www.lapatria.com/internacional/8-anos-despues-condenan-responsables-de-la-muerte-de-41-ninas-en-un-incendio-aun-hay. Omitiendo.
  > Scrapeando artículo: https://www.lapatria.com/opinion/columnistas/gonzalo-gallo
    * Error de tiempo de espera al cargar el artículo: https://www.lapatria.com/opinion/columnistas/gonzalo-gallo. Omitiendo.
  - URL de sección detectada: https://www.lapatria.com/nacional. Omitiendo.
  > Scrapeando artículo: https://www.lapatria.com/manizales/aumenta-el-uso-de-tusi-en-manizales-preocupaciones-y-desafios-actuales
  > Scrapeando artículo: https://www.lapatria.com/salud/tusi-una-combinacion-que-llega-ser-letal-para-el-organismo
  > Scrapeando artículo: https:

**Web scraping: La opinión**

In [2]:
# --- CONFIGURACIÓN INICIAL ---
URL_BASE = "https://www.laopinion.co"
URL_BUSQUEDA = "https://www.laopinion.co/search/node?keys=fentanilo&_wrapper_format=html"
PALABRA_CLAVE = "fentanilo"
PAGINA_INICIO = 0
PAGINA_FINAL = 19
TIMEOUT = 30

def limpiar_texto(soup_articulo):
    """Extrae y limpia el texto del cuerpo de la noticia de La Opinión."""
    
    # Encontrar el contenedor principal del artículo
    cuerpo_contenedor = soup_articulo.find('article')
    
    if not cuerpo_contenedor:
        return ""
    
    # Lista de patrones de texto y clases a eliminar
    patrones_a_eliminar = [
        r'Ver Contenidos Relacionados al Autor',
        r'Authored by\s?cherrera',
        r'Authored by cherrera',
        r'ImageLa Opinión',
        r'Categoría notaPremium',
        r'Este contenido esSi quieres disfrutar el artículo completo te invitamos a suscribirte.*',
        r'Plan Básico Digital Mensual Pago mensual.*',
        r'Plan Básico Digital Trimestral Pago cada 3 meses.*',
        r'Plan Básico Digital Anual Pago anual.*',
        r'Conoce todos nuestros planes disponiblesaquí ¿Ya eres suscriptor\?Inicia Sesión ¿Ya eres suscriptor\?',
        r'Tu apoyo para seguir manteniendo el enfoque.*',
        r'Suscribirme.*',
        r'Recomendados para ti.*',
        r'AutorLeonardo Favio OliverosLo que dijo Bernal sobre el aeropuerto de Cúcuta al anunciar que el de San Antonio será ‘de lujo’',
        r'AutorGustavo Contreras SabogalCúcuta Deportivo y Medellín chocarán en los cuartos de final de la Copa Colombia',
        r'AutorLa OpiniónAbrirán centro de bienestar emocional en Cúcuta para pacientes con cáncer',
        r'Consulte:.*',
        r'Lea:.*',
        r'Le puede interesar:.*',
        r'Gracias por valorar.*La Opinión Digital.*'
    ]
    
    # Se crea una nueva lista para almacenar el texto final
    cuerpo_final = []
    
    # Iterar sobre todos los párrafos y divs que podrían contener el texto
    for element in cuerpo_contenedor.find_all(['p', 'div']):
        text = element.get_text(strip=True)
        # Si el texto no está vacío y no coincide con ningún patrón a eliminar, se añade a la lista final
        if text and not any(re.search(p, text, re.IGNORECASE) for p in patrones_a_eliminar):
            cuerpo_final.append(text)
    
    return " ".join(cuerpo_final)

def scrap_articulo(url, driver):
    """Navega a la URL de un artículo y extrae todos los detalles."""
    print(f"  > Scrapeando artículo: {url}")
    try:
        driver.get(url)
        WebDriverWait(driver, TIMEOUT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'div.header-content'))
        )
        time.sleep(2)
        
        soup_articulo = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Extracción de datos
        titulo_tag = soup_articulo.find('div', class_='views-field-title')
        titulo = titulo_tag.get_text(strip=True) if titulo_tag else 'No encontrado'
        
        autor_tag = soup_articulo.find('div', class_='field--name-field-metaarticle-author')
        autor = autor_tag.get_text(strip=True) if autor_tag else 'No encontrado'
        
        fecha_tag = soup_articulo.find('span', class_='field--name-created')
        fecha = fecha_tag.get_text(strip=True) if fecha_tag else 'No encontrada'
        
        cuerpo = limpiar_texto(soup_articulo)
        
        texto_completo = f"{titulo} {cuerpo}".lower()
        conteo = len(re.findall(r'\b' + PALABRA_CLAVE.lower() + r'\b', texto_completo))
        
        return {
            'título': titulo,
            'cuerpo_noticia': cuerpo,
            'autor': autor,
            'fecha': fecha,
            'url': url,
            'conteo_fentanilo': conteo
        }
    except TimeoutException:
        print(f"    * Error de tiempo de espera al cargar el artículo: {url}. Omitiendo.")
        return None
    except Exception as e:
        print(f"    * Error al scrapear el artículo {url}: {e}")
        return None
    finally:
        try:
            driver.back()
        except:
            pass

# --- INICIO DEL SCRIPT PRINCIPAL ---
GECKODRIVER_PATH = os.path.join(os.getcwd(), 'geckodriver.exe')
LIBREWOLF_PATH = r"C:\Program Files\LibreWolf\librewolf.exe"
print("Iniciando el navegador LibreWolf...")
options = FirefoxOptions()
options.add_argument('--headless')
options.add_argument('--log-level=3')
options.binary_location = LIBREWOLF_PATH
service = Service(GECKODRIVER_PATH)
driver = webdriver.Firefox(service=service, options=options)

articulos_recopilados = []
urls_vistas = set()

try:
    for pagina_actual in range(PAGINA_INICIO, PAGINA_FINAL + 1):
        url_paginada = f"{URL_BUSQUEDA}&page={pagina_actual}"
        print(f"\n--- Procesando página de resultados #{pagina_actual} ---")
        
        driver.get(url_paginada)
        try:
            WebDriverWait(driver, TIMEOUT).until(
                EC.presence_of_element_located((By.ID, 'main_search'))
            )
            time.sleep(3)
            
            soup_resultados = BeautifulSoup(driver.page_source, 'html.parser')
            resultados = soup_resultados.find_all('div', class_='row node_search-results')
            
            if not resultados:
                print("No se encontraron más resultados. Finalizando.")
                break
            
            print(f"Se encontraron {len(resultados)} resultados en la página.")
            for item in resultados:
                url_articulo_tag = item.find('a', href=True)
                if url_articulo_tag:
                    url_articulo = url_articulo_tag['href']
                    if not url_articulo.startswith('http'):
                        url_articulo = f"https://www.laopinion.co{url_articulo}"

                    if url_articulo not in urls_vistas:
                        urls_vistas.add(url_articulo)
                        datos_articulo = scrap_articulo(url_articulo, driver)
                        time.sleep(3)
                        if datos_articulo:
                            articulos_recopilados.append(datos_articulo)
                    else:
                        print(f"  - URL duplicada detectada: {url_articulo}. Omitiendo.")
                else:
                    print("  - No se encontró URL en el resultado. Saltando.")
        
        except TimeoutException:
            print("Error de tiempo de espera al cargar la página de resultados. Intentando continuar.")
        
        if pagina_actual % 5 == 0 and articulos_recopilados:
            df = pd.DataFrame(articulos_recopilados)
            df.to_csv(f'noticias_fentanilo_laopinion_pag_{pagina_actual}.csv', index=False, encoding='utf-8-sig')
            print(f"\nSe han guardado los datos de las últimas 5 páginas en el archivo 'noticias_fentanilo_laopinion_pag_{pagina_actual}.csv'.")
            articulos_recopilados = []

finally:
    print("\nCerrando el navegador.")
    driver.quit()

if articulos_recopilados:
    df = pd.DataFrame(articulos_recopilados)
    df.to_csv('noticias_fentanilo_laopinion_final.csv', index=False, encoding='utf-8-sig')
    print("Los datos restantes han sido guardados en el archivo 'noticias_fentanilo_laopinion_final.csv'.")
elif not os.path.exists('noticias_fentanilo_laopinion_final.csv') and not [f for f in os.listdir() if f.startswith('noticias_fentanilo_laopinion_pag_')]:
    print("\nNo se recopiló ningún artículo.")

print("\nProceso finalizado.")

Iniciando el navegador LibreWolf...

--- Procesando página de resultados #0 ---
Se encontraron 10 resultados en la página.
  > Scrapeando artículo: https://www.laopinion.co/premium/judicial/el-fentanilo-lo-estan-combinando-con-heroina-en-cucuta-y-ocana
  > Scrapeando artículo: https://www.laopinion.co/sucesos/el-fentanilo-sigue-preocupando-en-colombia
  > Scrapeando artículo: https://www.laopinion.co/judicial/siguen-las-incautaciones-de-fentanilo-en-cucuta
  > Scrapeando artículo: https://www.laopinion.co/mundo/no-creemos-que-el-fentanilo-este-reemplazando-la-cocaina-estados-unidos
  > Scrapeando artículo: https://www.laopinion.co/mundo/fentanilo-detras-de-la-peor-crisis-de-drogas-en-la-historia-de-ee-uu
  > Scrapeando artículo: https://www.laopinion.co/salud/que-es-el-fentanilo-esa-droga-peligrosa-y-adictiva
  > Scrapeando artículo: https://www.laopinion.co/salud/que-habria-detras-de-las-sobredosis-por-fentanilo-y-alcohol
  > Scrapeando artículo: https://www.laopinion.co/colombia/tusi